## sellers

In [0]:
# Inspeção da tabela bronze.sellers antes da transformação:
# estrutura (schema) e amostra dos dados para orientar tipagem e padronização.
df_sellers_bronze = spark.table("ecommerce_br.bronze.sellers")

df_sellers_bronze.printSchema()
df_sellers_bronze.show(5, truncate=False)

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+--------------------------------+----------------------+-----------------+------------+--------------------------+-------------------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|data_ingestao             |arquivo_origem           |
+--------------------------------+----------------------+-----------------+------------+--------------------------+-------------------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |2026-09-07 18:24:45.123856|olist_sellers_dataset.csv|
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |2026-09-07 18:24:45.123856|olist_sellers_dataset.csv|
|ce3ad9d

In [0]:
from pyspark.sql.functions import col, upper, trim, current_timestamp

# Transformação Silver da tabela sellers.
# seller_id e seller_zip_code_prefix permanecem como texto: são identificadores/códigos
# (o CEP tem zeros à esquerda que se perderiam se convertidos para número).
# Cidade padronizada para maiúsculas e sem espaços laterais, garantindo consistência
# em futuras junções. Estado mantido como veio (já uniforme).
# data_processamento_silver: carimbo de linhagem — momento em que a camada Silver foi gerada.
df_sellers_silver = df_sellers_bronze.select(
    trim(col("seller_id")).alias("seller_id"),
    trim(col("seller_zip_code_prefix")).alias("seller_zip_code_prefix"),
    upper(trim(col("seller_city"))).alias("seller_city"),
    upper(trim(col("seller_state"))).alias("seller_state")
).dropDuplicates() \
 .withColumn("data_processamento_silver", current_timestamp())

df_sellers_silver.show(5, truncate=False)
print(f"Total de vendedores (Silver): {df_sellers_silver.count()}")

+--------------------------------+----------------------+-----------------+------------+--------------------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|data_processamento_silver |
+--------------------------------+----------------------+-----------------+------------+--------------------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |CAMPINAS         |SP          |2026-09-17 01:28:11.736046|
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |MOGI GUACU       |SP          |2026-09-17 01:28:11.736046|
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |RIO DE JANEIRO   |RJ          |2026-09-17 01:28:11.736046|
|c0f3eea2e14555b6faeea3dd58c1b1c3|04195                 |SAO PAULO        |SP          |2026-09-17 01:28:11.736046|
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |BRAGANCA PAULISTA|SP          |2026-09-17 01:28:11.736046|
+--------------------------------+----------------------+---------------

In [0]:
# Persistência da tabela tratada no schema silver.
# mode overwrite + overwriteSchema: escrita idempotente, coerente com o padrão do pipeline.
df_sellers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.sellers")

print("Tabela ecommerce_br.silver.sellers criada com sucesso!")

Tabela ecommerce_br.silver.sellers criada com sucesso!


## customers — clientes e localização

In [0]:
# Inspeção da tabela bronze.customers antes da transformação.
df_customers_bronze = spark.table("ecommerce_br.bronze.customers")

df_customers_bronze.printSchema()
df_customers_bronze.show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+---------------------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|data_ingestao             |arquivo_origem             |
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+---------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca       

In [0]:
from pyspark.sql.functions import col, upper, trim, current_timestamp

# Transformação Silver da tabela customers.
# Dois identificadores distintos, ambos mantidos como texto:
#   customer_id: identifica o cliente em cada pedido (chave de ligação com orders).
#   customer_unique_id: identifica a pessoa real (mesmo cliente em compras diferentes
#   compartilha este id) — essencial para distinguir clientes recorrentes.
# CEP como texto (preserva zeros à esquerda). Cidade em maiúsculas e sem espaços laterais.
df_customers_silver = df_customers_bronze.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_unique_id")).alias("customer_unique_id"),
    trim(col("customer_zip_code_prefix")).alias("customer_zip_code_prefix"),
    upper(trim(col("customer_city"))).alias("customer_city"),
    upper(trim(col("customer_state"))).alias("customer_state")
).dropDuplicates() \
 .withColumn("data_processamento_silver", current_timestamp())

df_customers_silver.show(5, truncate=False)
print(f"Total de linhas (Silver customers): {df_customers_silver.count()}")

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|data_processamento_silver |
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+--------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |FRANCA               |SP            |2026-09-17 01:28:19.313003|
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|09790                   |SAO BERNARDO DO CAMPO|SP            |2026-09-17 01:28:19.313003|
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|01151                   |SAO PAULO            |SP            |2026-09-17 01:28:19.313003|
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9ac

In [0]:
# Verificação exploratória: total de registros vs. pessoas únicas.
# customer_id conta clientes-por-pedido; customer_unique_id conta pessoas reais.
# A diferença revela a proporção de clientes recorrentes na base.
total_registros = df_customers_silver.count()
clientes_unicos = df_customers_silver.select("customer_unique_id").distinct().count()

print(f"Registros (customer_id):        {total_registros}")
print(f"Clientes únicos (unique_id):    {clientes_unicos}")
print(f"Diferença (compras recorrentes): {total_registros - clientes_unicos}")

Registros (customer_id):        99441
Clientes únicos (unique_id):    96096
Diferença (compras recorrentes): 3345


In [0]:
df_customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.customers")

print("Tabela ecommerce_br.silver.customers criada com sucesso!")

Tabela ecommerce_br.silver.customers criada com sucesso!


## products — produtos e categorias

In [0]:
# Inspeção da tabela bronze.products antes da transformação.
df_products_bronze = spark.table("ecommerce_br.bronze.products")

df_products_bronze.printSchema()
df_products_bronze.show(5, truncate=False)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------------+--------------------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|data_ingestao

In [0]:
from pyspark.sql.functions import col

# Verificação de completude: contagem de nulos por coluna relevante.
# Identifica onde há dados faltantes antes de decidir o tratamento.
total = df_products_bronze.count()
print(f"Total de produtos: {total}\n")

for coluna in ["product_category_name", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    nulos = df_products_bronze.filter(col(coluna).isNull()).count()
    print(f"{coluna}: {nulos} nulos ({nulos/total*100:.2f}%)")

Total de produtos: 32951

product_category_name: 610 nulos (1.85%)
product_weight_g: 2 nulos (0.01%)
product_length_cm: 2 nulos (0.01%)
product_height_cm: 2 nulos (0.01%)
product_width_cm: 2 nulos (0.01%)


In [0]:
from pyspark.sql.functions import col, trim, current_timestamp, coalesce, lit

# Transformação Silver da tabela products — parte 1: tipagem e tratamento de nulos.
# product_id permanece como texto (identificador).
# Colunas de contagem e medidas convertidas para inteiro (são quantidades, não códigos).
# Categoria nula recebe rótulo explícito "indefinida": preserva o produto nas análises
# por categoria em vez de descartá-lo ou deixar nulo solto.
# Medidas/peso nulos (2 registros) permanecem nulos: ausência de medida não é zero.
df_products_silver = df_products_bronze.select(
    trim(col("product_id")).alias("product_id"),
    coalesce(trim(col("product_category_name")), lit("indefinida")).alias("product_category_name"),
    col("product_name_lenght").cast("int").alias("product_name_lenght"),
    col("product_description_lenght").cast("int").alias("product_description_lenght"),
    col("product_photos_qty").cast("int").alias("product_photos_qty"),
    col("product_weight_g").cast("int").alias("product_weight_g"),
    col("product_length_cm").cast("int").alias("product_length_cm"),
    col("product_height_cm").cast("int").alias("product_height_cm"),
    col("product_width_cm").cast("int").alias("product_width_cm")
)

df_products_silver.printSchema()
df_products_silver.show(5, truncate=False)

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = false)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-------------

In [0]:
# Inspeção da tabela de tradução de categorias (apoio para o join).
df_categorias = spark.table("ecommerce_br.bronze.category_translation")
df_categorias.printSchema()
df_categorias.show(5, truncate=False)

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+----------------------+-----------------------------+--------------------------+-------------------------------------+
|product_category_name |product_category_name_english|data_ingestao             |arquivo_origem                       |
+----------------------+-----------------------------+--------------------------+-------------------------------------+
|beleza_saude          |health_beauty                |2026-09-07 18:24:47.965993|product_category_name_translation.csv|
|informatica_acessorios|computers_accessories        |2026-09-07 18:24:47.965993|product_category_name_translation.csv|
|automotivo            |auto                         |2026-09-07 18:24:47.965993|product_category_name_translation.csv|
|cama_mesa_banho       |bed_bath_table               |2026-09-07 1

In [0]:
from pyspark.sql.functions import col, trim, current_timestamp, coalesce, lit

# Tabela de tradução base (categorias PT -> EN), apenas as colunas úteis.
df_traducao = df_categorias.select("product_category_name", "product_category_name_english")

# Complemento manual: duas categorias reais ausentes na tradução oficial do Olist,
# identificadas na conferência de integridade (13 produtos afetados).
# Corrige a lacuna na origem, antes do join, mantendo o padrão de nomes em inglês.
traducoes_faltantes = spark.createDataFrame(
    [
        ("pc_gamer", "pc_gamer"),
        ("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_food_preparers"),
    ],
    ["product_category_name", "product_category_name_english"]
)

df_traducao = df_traducao.union(traducoes_faltantes)

# Left join preserva todos os produtos; categorias genuinamente sem categoria
# ("indefinida") permanecem sem tradução e recebem o rótulo "undefined".
df_products_silver = df_products_silver.join(
    df_traducao,
    on="product_category_name",
    how="left"
).withColumn(
    "product_category_name_english",
    coalesce(col("product_category_name_english"), lit("undefined"))
).withColumn("data_processamento_silver", current_timestamp())

df_products_silver.printSchema()
print(f"Total de produtos (Silver): {df_products_silver.count()}")

root
 |-- product_category_name: string (nullable = false)
 |-- product_id: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- product_category_name_english: string (nullable = false)
 |-- data_processamento_silver: timestamp (nullable = false)

Total de produtos (Silver): 32951


In [0]:
# Reconferência pós-correção: 'undefined' deve restar apenas para categorias
# genuinamente ausentes ("indefinida"), não mais por lacuna de tradução.
undefined_total = df_products_silver.filter(col("product_category_name_english") == "undefined").count()
print(f"Produtos com tradução 'undefined': {undefined_total}")

# Confirma que as duas categorias corrigidas agora têm tradução.
df_products_silver.filter(
    col("product_category_name").isin("pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos")
).select("product_category_name", "product_category_name_english").distinct().show(truncate=False)

Produtos com tradução 'undefined': 610
+---------------------------------------------+-------------------------------+
|product_category_name                        |product_category_name_english  |
+---------------------------------------------+-------------------------------+
|pc_gamer                                     |pc_gamer                       |
|portateis_cozinha_e_preparadores_de_alimentos|portable_kitchen_food_preparers|
+---------------------------------------------+-------------------------------+



In [0]:
# Persistência da tabela tratada no schema silver.
df_products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.products")

print("Tabela ecommerce_br.silver.products criada com sucesso!")

Tabela ecommerce_br.silver.products criada com sucesso!


## order_payments — pagamentos

In [0]:
# Inspeção da tabela bronze.order_payments antes da transformação.
df_payments_bronze = spark.table("ecommerce_br.bronze.order_payments")

df_payments_bronze.printSchema()
df_payments_bronze.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+--------------------------------+------------------+------------+--------------------+-------------+--------------------------+--------------------------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|data_ingestao             |arquivo_origem                  |
+--------------------------------+------------------+------------+--------------------+-------------+--------------------------+--------------------------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |2026-09-07 18:24:34.330329|olist_order_payments_dataset.csv|
|a9810da82917af2d9aef

In [0]:
from pyspark.sql.functions import col, trim, lower, current_timestamp

# Transformação Silver da tabela order_payments.
# order_id permanece como texto (chave de ligação com orders).
# payment_sequential e payment_installments são contagens -> inteiro.
# payment_value é valor monetário -> decimal(10,2): precisão exata em centavos,
# evitando o erro de arredondamento do ponto flutuante (double) em somas financeiras.
# payment_type mantido em minúsculas (valor já padronizado na origem), apenas com trim.
df_payments_silver = df_payments_bronze.select(
    trim(col("order_id")).alias("order_id"),
    col("payment_sequential").cast("int").alias("payment_sequential"),
    lower(trim(col("payment_type"))).alias("payment_type"),
    col("payment_installments").cast("int").alias("payment_installments"),
    col("payment_value").cast("decimal(10,2)").alias("payment_value")
).dropDuplicates() \
 .withColumn("data_processamento_silver", current_timestamp())

df_payments_silver.printSchema()
df_payments_silver.show(5, truncate=False)
print(f"Total de pagamentos (Silver): {df_payments_silver.count()}")

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(10,2) (nullable = true)
 |-- data_processamento_silver: timestamp (nullable = false)

+--------------------------------+------------------+------------+--------------------+-------------+--------------------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|data_processamento_silver |
+--------------------------------+------------------+------------+--------------------+-------------+--------------------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |2026-09-17 01:28:43.803184|
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |1                   |24.39        |2026-09-17 01:28:43.803184|
|25e8ea4e93396b6fa0d3dd708e76c1bd|1             

In [0]:
# Persistência da tabela tratada no schema silver.
df_payments_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.order_payments")

print("Tabela ecommerce_br.silver.order_payments criada com sucesso!")

Tabela ecommerce_br.silver.order_payments criada com sucesso!


## order_items — itens do pedido

In [0]:
# Inspeção da tabela bronze.order_items antes da transformação.
df_items_bronze = spark.table("ecommerce_br.bronze.order_items")

df_items_bronze.printSchema()
df_items_bronze.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+--------------------------+-----------------------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|data_ingestao             |arquivo_origem               |
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+------------------

In [0]:
from pyspark.sql.functions import col, trim, current_timestamp

# Transformação Silver da tabela order_items.
# Chaves (order_id, product_id, seller_id) permanecem como texto.
# order_item_id é o sequencial do item no pedido -> inteiro.
# shipping_limit_date convertida para timestamp (formato ISO reconhecido nativamente),
# habilitando cálculos temporais.
# price e freight_value são valores monetários -> decimal(10,2) para precisão em centavos.
df_items_silver = df_items_bronze.select(
    trim(col("order_id")).alias("order_id"),
    col("order_item_id").cast("int").alias("order_item_id"),
    trim(col("product_id")).alias("product_id"),
    trim(col("seller_id")).alias("seller_id"),
    col("shipping_limit_date").cast("timestamp").alias("shipping_limit_date"),
    col("price").cast("decimal(10,2)").alias("price"),
    col("freight_value").cast("decimal(10,2)").alias("freight_value")
).dropDuplicates() \
 .withColumn("data_processamento_silver", current_timestamp())

df_items_silver.printSchema()
df_items_silver.show(5, truncate=False)
print(f"Total de itens (Silver): {df_items_silver.count()}")

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- freight_value: decimal(10,2) (nullable = true)
 |-- data_processamento_silver: timestamp (nullable = false)

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+--------------------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|data_processamento_silver |
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+--------------------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb49

In [0]:
# Verificação da conversão de data: datas em formato inesperado viram null.
# Como o Bronze tinha shipping_limit_date sempre preenchida, nulos aqui indicariam
# falha de conversão, não ausência de dado.
nulos_data = df_items_silver.filter(col("shipping_limit_date").isNull()).count()
print(f"shipping_limit_date nulos após conversão: {nulos_data}")

shipping_limit_date nulos após conversão: 0


In [0]:
# Persistência da tabela tratada no schema silver.
df_items_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.order_items")

print("Tabela ecommerce_br.silver.order_items criada com sucesso!")

Tabela ecommerce_br.silver.order_items criada com sucesso!


## orders — pedidos (tabela central)

In [0]:
# Inspeção da tabela bronze.orders antes da transformação.
df_orders_bronze = spark.table("ecommerce_br.bronze.orders")

df_orders_bronze.printSchema()
df_orders_bronze.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------------+------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|data_

In [0]:
# Distribuição dos status de pedido: contextualiza os nulos nas datas
# (ex.: pedidos não entregues não têm data de entrega ao cliente).
df_orders_bronze.groupBy("order_status").count().orderBy(col("count").desc()).show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



In [0]:
# Completude das 5 colunas de data.
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

total = df_orders_bronze.count()
print(f"Total de pedidos: {total}\n")
for c in colunas_data:
    n = df_orders_bronze.filter(col(c).isNull()).count()
    print(f"{c}: {n} nulos ({n/total*100:.2f}%)")

Total de pedidos: 99441

order_purchase_timestamp: 0 nulos (0.00%)
order_approved_at: 160 nulos (0.16%)
order_delivered_carrier_date: 1783 nulos (1.79%)
order_delivered_customer_date: 2965 nulos (2.98%)
order_estimated_delivery_date: 0 nulos (0.00%)


In [0]:
from pyspark.sql.functions import col, trim, lower, current_timestamp

# Transformação Silver da tabela orders.
# order_id e customer_id permanecem como texto (chaves).
# order_status padronizado em minúsculas (valores de sistema).
# As 5 colunas de data convertidas para timestamp (formato ISO nativo).
# Nulos nas datas são PRESERVADOS: representam o funil de entrega (pedidos não
# aprovados/enviados/entregues). Não são erro — filtrar entregues é decisão da análise.
df_orders_silver = df_orders_bronze.select(
    trim(col("order_id")).alias("order_id"),
    trim(col("customer_id")).alias("customer_id"),
    lower(trim(col("order_status"))).alias("order_status"),
    col("order_purchase_timestamp").cast("timestamp").alias("order_purchase_timestamp"),
    col("order_approved_at").cast("timestamp").alias("order_approved_at"),
    col("order_delivered_carrier_date").cast("timestamp").alias("order_delivered_carrier_date"),
    col("order_delivered_customer_date").cast("timestamp").alias("order_delivered_customer_date"),
    col("order_estimated_delivery_date").cast("timestamp").alias("order_estimated_delivery_date")
).dropDuplicates() \
 .withColumn("data_processamento_silver", current_timestamp())

df_orders_silver.printSchema()
print(f"Total de pedidos (Silver): {df_orders_silver.count()}")

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- data_processamento_silver: timestamp (nullable = false)

Total de pedidos (Silver): 99441


In [0]:
# Verificação pós-conversão: os nulos devem bater com os da Bronze.
# Divergência indicaria falha de parsing (data válida virando null), não nulo legítimo.
for c in colunas_data:
    n = df_orders_silver.filter(col(c).isNull()).count()
    print(f"{c}: {n} nulos")

order_purchase_timestamp: 0 nulos
order_approved_at: 160 nulos
order_delivered_carrier_date: 1783 nulos
order_delivered_customer_date: 2965 nulos
order_estimated_delivery_date: 0 nulos


In [0]:
# Persistência da tabela tratada no schema silver.
df_orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.orders")

print("Tabela ecommerce_br.silver.orders criada com sucesso!")

Tabela ecommerce_br.silver.orders criada com sucesso!


## geolocation — coordenadas por CEP

In [0]:
# Inspeção da tabela bronze.geolocation antes da transformação.
df_geo_bronze = spark.table("ecommerce_br.bronze.geolocation")

df_geo_bronze.printSchema()
df_geo_bronze.show(5, truncate=False)

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: string (nullable = true)
 |-- geolocation_lng: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+---------------------------+-------------------+------------------+----------------+-----------------+--------------------------+-----------------------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng   |geolocation_city|geolocation_state|data_ingestao             |arquivo_origem               |
+---------------------------+-------------------+------------------+----------------+-----------------+--------------------------+-----------------------------+
|01037                      |-23.54562128115268 |-46.63929204800168|sao paulo       |SP               |2026-09-07 18:24:19.811831|olist_geolocation_dataset.csv|
|01046

In [0]:
# Dimensão do problema: total de linhas vs. CEPs únicos.
# Confirma a multiplicidade de coordenadas por prefixo que a agregação vai resolver.
total_linhas = df_geo_bronze.count()
ceps_unicos = df_geo_bronze.select("geolocation_zip_code_prefix").distinct().count()
print(f"Total de linhas: {total_linhas}")
print(f"CEPs únicos: {ceps_unicos}")
print(f"Média de pontos por CEP: {total_linhas/ceps_unicos:.1f}")

Total de linhas: 1000163
CEPs únicos: 19015
Média de pontos por CEP: 52.6


In [0]:
from pyspark.sql.functions import col, trim, avg, current_timestamp

# Transformação Silver da tabela geolocation.
# Cada prefixo de CEP tem múltiplas coordenadas (média de 52,6 pontos/CEP).
# A agregação reduz a um ponto representativo por CEP: a média de latitude e longitude,
# viabilizando o cálculo de distância cliente x vendedor (pergunta 5).
# lat/lng convertidos para double (coordenadas são medidas científicas, não valores
# monetários) antes da média. Tabela enxuta: apenas CEP + coordenadas médias.
df_geo_silver = df_geo_bronze.select(
    trim(col("geolocation_zip_code_prefix")).alias("geolocation_zip_code_prefix"),
    col("geolocation_lat").cast("double").alias("geolocation_lat"),
    col("geolocation_lng").cast("double").alias("geolocation_lng")
).groupBy("geolocation_zip_code_prefix").agg(
    avg("geolocation_lat").alias("geolocation_lat"),
    avg("geolocation_lng").alias("geolocation_lng")
).withColumn("data_processamento_silver", current_timestamp())

df_geo_silver.printSchema()
df_geo_silver.show(5, truncate=False)
print(f"Total de CEPs (Silver): {df_geo_silver.count()}")

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- data_processamento_silver: timestamp (nullable = false)

+---------------------------+-------------------+-------------------+--------------------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng    |data_processamento_silver |
+---------------------------+-------------------+-------------------+--------------------------+
|01037                      |-23.545411016454086|-46.63892437204837 |2026-09-17 01:29:10.284026|
|01046                      |-23.545930039168788|-46.6439975177819  |2026-09-17 01:29:10.284026|
|01041                      |-23.54390458746036 |-46.640046486444724|2026-09-17 01:29:10.284026|
|01035                      |-23.541614488738958|-46.64154914252655 |2026-09-17 01:29:10.284026|
|01012                      |-23.54778973314686 |-46.63485900505736 |2026-09-17 01:29:10.284026|
+---

In [0]:
# Persistência da tabela agregada no schema silver.
df_geo_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.geolocation")

print("Tabela ecommerce_br.silver.geolocation criada com sucesso!")

Tabela ecommerce_br.silver.geolocation criada com sucesso!


## order_reviews — avaliações

In [0]:
# Inspeção da tabela bronze.order_reviews antes da transformação.
df_reviews_bronze = spark.table("ecommerce_br.bronze.order_reviews")

df_reviews_bronze.printSchema()
df_reviews_bronze.show(5, truncate=False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- arquivo_origem: string (nullable = true)

+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+--------------------------+-------------------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|data_ing

In [0]:
from pyspark.sql.functions import col, trim, current_timestamp

# Transformação Silver da tabela order_reviews.
# review_id e order_id permanecem como texto (chaves).
# review_score convertido para inteiro (nota de 1 a 5) — métrica central das análises
# de satisfação (P1, P2, P3).
# Comentários (título e mensagem) mantidos como texto, apenas com trim; nulos são
# esperados e preservados (nem toda avaliação tem comentário escrito).
# As duas datas convertidas para timestamp (formato ISO nativo).
df_reviews_silver = df_reviews_bronze.select(
    trim(col("review_id")).alias("review_id"),
    trim(col("order_id")).alias("order_id"),
    col("review_score").cast("int").alias("review_score"),
    trim(col("review_comment_title")).alias("review_comment_title"),
    trim(col("review_comment_message")).alias("review_comment_message"),
    col("review_creation_date").cast("timestamp").alias("review_creation_date"),
    col("review_answer_timestamp").cast("timestamp").alias("review_answer_timestamp")
).dropDuplicates() \
 .withColumn("data_processamento_silver", current_timestamp())

df_reviews_silver.printSchema()
df_reviews_silver.show(5, truncate=False)
print(f"Total de avaliações (Silver): {df_reviews_silver.count()}")

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- data_processamento_silver: timestamp (nullable = false)

+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+--------------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|data_processamento_silver |
+--------------------------------+----

In [0]:
# Validação de domínio da nota: review_score deve conter apenas valores de 1 a 5.
# Valores fora do intervalo ou nulos indicariam problema de qualidade a tratar.
df_reviews_silver.groupBy("review_score").count().orderBy("review_score").show()

+------------+-----+
|review_score|count|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19142|
|           5|57328|
+------------+-----+



In [0]:
# Persistência da tabela tratada no schema silver.
df_reviews_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_br.silver.order_reviews")

print("Tabela ecommerce_br.silver.order_reviews criada com sucesso!")

Tabela ecommerce_br.silver.order_reviews criada com sucesso!
